# Running MD with pmemd
https://ambermd.org/tutorials/basic/tutorial14/index.php

Short simulation (50 ns) of a protein system in a truncated octahedral box solvated in explicit water

Continuation of `03_`

NOTE: have to think of how to simulate with the stabilizer (FSC)...

NOTE: all bash commands run in the terminal of `/home/wjoon21/project/chen2023_possible_allosteric/B2/trial5_tleap_script_parm7_rst7/`

## Input files:
1. A topology file generated by LEaP (`amber_o98_ion.parm7`) [final output of `02_`]
2. A coordinate file of the equilibrated system (`9md.rst7`) [final output of `03_`]
3. A `md.in` input file (below)

`md.in`
```in
Explicit solvent molecular dynamics constant pressure 50 ns MD
 &cntrl
   imin=0, irest=1, ntx=5, 
   ntpr=500000, ntwx=500000, ntwr=500000, nstlim=25000000, 
   dt=0.002, ntt=3, tempi=300, 
   temp0=300, gamma_ln=1.0, ig=-1, 
   ntp=1, ntc=2, ntf=2, cut=9, 
   ntb=2, iwrap=1, ioutfm=1, 
/ 

```
Important settings:
1. `nstlim`\
Controls how long your simulation is by telling the program how many steps of MD to run. Multiplication of the time step by the number of steps will give you the length of the simulation in ps. Here, 25,000,000 steps translates to 50 ns of real-time simulation. If you want to run for a different amount of time, you will have to change the value of nstlim.

2. `ntpr`, `ntwx` and `ntwr`\
Control how often a frame of the simulation is saved. Here, `ntpr=500000` translates to saving every 500,000 steps or 1 ns in the `md.out` file. The ntwx namelist parameter refers to how often the frame will be saved to the restart file. The ntwr namelist parameter writes to the trajectory file (`.nc`). You may need to adjust this parameter to write out more often if you are looking at finer changes.

3. `dt`\
This variable is the time step in picoseconds (1 ps = 10^{-12} s). The time step is basically the time unit of MD simulations. Here, it is 0.002 ps, which is 2.0 fs. As for choosing what the time step of your simulation should be, it depends on the temperature and if the SHAKE algorithm is used. Here, we do use the SHAKE algorithm (`ntb=2`, `btc=2`, and `ntf=2`), so the maximum time step should be 0.002 (0.001 if the SHAKE algorithm is not used). For temperatures above 300 K, the time step should be reduced. This is because the increased velocity will lead to longer distances travelled in between force evaluations, which could cause the system to blow up.

4. `ntt`\
This variable defines which thermostat is used. Here, we use the Langevin thermostat (`ntt=3`). Langevin dynamics are susceptible to "synchronization" artifacts, so the ig variable needs to be specifically set to counter this. It is recommended that `ig` be set to `-1` (the default) for `ntt=3`. When `ig=-1`, the random seed is based on the current time and date, so it will be different for each run.

5. `ntp`\
This is the variable that controls pressure dynamics. Because we want to run our simulation under conditions of constant pressure, it should be set equal to 1 (which corresponds to MD with isotropic position scaling). ~~If the system is in an orthogonal box (meaning all angles are 90 degrees), then it should be set to 2.~~ (AMBER22 manual: "`ntp = 2` md with anisotropic (x-,y-,z-) pressure scaling: this should only be used with orthogonal boxes (i.e. with all angles set to 90 degrees). Anisotropic scaling is primarily intended for non-isotropic systems, such as membrane simulations, where the surface tensions are different in different directions; it is *generally not appropriate for solutes dissolved in water*."). Because the box for this simulation is a truncated octahedron, we should set it equal to 1. The Berendsen barostat is used by default.

In [1]:
25000000 * 0.002

50000.0

In [2]:
1250000 * 0.004

5000.0

## Running production MD

`prod_unrestrained_md_jobfile.sh`
```sh
#! /bin/bash -f

export CUDA_VISIBLE_DEVICES=0 # run on the GPU designated 0 (check below)

$AMBERHOME/bin/pmemd.cuda -O -i prod_unrestrained_md.in -p amber_o98_ion.parm7 -c 9md.rst7\
 -ref 9md.rst7 -o o98_prod_md.mdout -r o98_prod_md.rst7 -x o98_prod_md.nc
```
Flags:
```
-O   Overwrite output files
-i   input file (.in)
-p   topology file (.parm7)
-c   the starting coordinate file (.rst7)
-ref reference coordinates; use the same as the starting coordinates
-o   output file (.out)
-r   restart file (last set of xyz coordinates from the simulation)
-x   file with trajectory (.nc) 
```
Terminal command:
```sh
bash prod_unrestrained_md_jobfile.sh &
```
`&` makes the job run in the background $\rightarrow$ can continue to use the command line

Check GPU designation in machine
```sh
nvidia-smi
```
```
Thu Jun 13 15:28:35 2024       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 495.29.05    Driver Version: 495.29.05    CUDA Version: 11.5     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA GeForce ...  On   | 00000000:01:00.0  On |                  N/A |
|  0%   50C    P2    36W / 151W |   7331MiB /  8116MiB |      1%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-----------------------------------------------------------------------------+
| Processes:                                                                  |
|  GPU   GI   CI        PID   Type   Process name                  GPU Memory |
|        ID   ID                                                   Usage      |
|=============================================================================|
|    0   N/A  N/A      1175      G   /usr/lib/xorg/Xorg                 53MiB |
|    0   N/A  N/A      1734      G   /usr/lib/xorg/Xorg                296MiB |
|    0   N/A  N/A      1869      G   /usr/bin/gnome-shell               59MiB |
|    0   N/A  N/A      2177      G   /usr/lib/firefox/firefox          385MiB |
|    0   N/A  N/A     10179      G   ...--variations-seed-version        7MiB |
|    0   N/A  N/A     12138      G   ...--variations-seed-version      125MiB |
|    0   N/A  N/A     88365    C+G   ...al/lib/vmd/vmd_LINUXAMD64     6378MiB |
+-----------------------------------------------------------------------------+
```
`0` is the only GPU available, and the model is
```sh
nvidia-smi --query-gpu=gpu_name --format=csv
```
```
name
NVIDIA GeForce GTX 1070
```

## MD output
SUCCESS (at least my system did NOT blow up...)

"blow up": infinite values (" `*******` " in Amber)

`o98_prod_md.mdout`
```
 ------------------------------------------------------------------------------


      A V E R A G E S   O V E R      50 S T E P S


 NSTEP = 25000000   TIME(PS) =   54000.000  TEMP(K) =   300.06  PRESS =     3.9
 Etot   =   -153421.5916  EKtot   =     26760.4780  EPtot      =   -180182.0696
 BOND   =       900.1423  ANGLE   =      2387.4669  DIHED      =      1595.8311
 UB     =         0.0000  IMP     =         0.0000  CMAP       =       416.4775
 1-4 NB =      1119.7306  1-4 EEL =     11341.2552  VDWAALS    =     20220.4270
 EELEC  =   -218163.4001  EHBOND  =         0.0000  RESTRAINT  =         0.0000
 EKCMT  =     11740.8146  VIRIAL  =     11704.4479  VOLUME     =    430571.6055
                                                    Density    =         1.0402
 ------------------------------------------------------------------------------


      R M S  F L U C T U A T I O N S


 NSTEP = 25000000   TIME(PS) =   54000.000  TEMP(K) =     1.46  PRESS =   126.9
 Etot   =       216.8424  EKtot   =       130.4227  EPtot      =       188.0998
 BOND   =        26.2747  ANGLE   =        41.0062  DIHED      =        18.1163
 UB     =         0.0000  IMP     =         0.0000  CMAP       =        25.1690
 1-4 NB =        13.5414  1-4 EEL =        52.7647  VDWAALS    =       193.3675
 EELEC  =       295.6695  EHBOND  =         0.0000  RESTRAINT  =         0.0000
 EKCMT  =        84.8270  VIRIAL  =      1149.9954  VOLUME     =       671.3270
                                                    Density    =         0.0016
 ------------------------------------------------------------------------------

--------------------------------------------------------------------------------
   5.  TIMINGS
--------------------------------------------------------------------------------

|  NonSetup CPU Time in Major Routines:
|
|     Routine           Sec        %
|     ------------------------------
|     Nonbond       62388.58   87.75
|     Bond              0.00    0.00
|     Angle             0.00    0.00
|     Dihedral          0.00    0.00
|     Shake            70.80    0.10
|     RunMD          8596.68   12.09
|     Other            45.67    0.06
|     ------------------------------
|     Total         71101.73

|  PME Nonbond Pairlist CPU Time:
|
|     Routine              Sec        %
|     ---------------------------------
|     Set Up Cit           0.00    0.00
|     Build List           0.00    0.00
|     ---------------------------------
|     Total                0.00    0.00

|  PME Direct Force CPU Time:
|
|     Routine              Sec        %
|     ---------------------------------
|     NonBonded Calc       0.00    0.00
|     Exclude Masked       0.00    0.00
|     Other               28.82    0.04
|     ---------------------------------
|     Total               28.82    0.04

|  PME Reciprocal Force CPU Time:
|
|     Routine              Sec        %
|     ---------------------------------
|     1D bspline           0.00    0.00
|     Grid Charges         0.00    0.00
|     Scalar Sum           0.00    0.00
|     Gradient Sum         0.00    0.00
|     FFT                  0.00    0.00
|     ---------------------------------
|     Total                0.00    0.00

|  Final Performance Info:
|     -----------------------------------------------------
|     Average timings for last       1 steps:
|     Elapsed(s) =       0.02 Per Step(ms) =      24.70
|         ns/day =       6.99   seconds/ns =   12351.93
|
|     Average timings for all steps:
|     Elapsed(s) =   71112.94 Per Step(ms) =       2.84
|         ns/day =      60.75   seconds/ns =    1422.26
|     -----------------------------------------------------

|  Setup CPU time:            0.79 seconds
|  NonSetup CPU time:     71101.73 seconds
|  Total CPU time:        71102.52 seconds    19.75 hours

|  Setup wall time:           2    seconds
|  NonSetup wall time:    71113    seconds
|  Total wall time:       71115    seconds    19.75 hours
```

`amber_o98_ion.parm7` + `o98_prod_md.nc`\
An example from frame 49 (0-index, 50 frames in total) showing apparent 
1. "detachment" of smaller protein from the larger protein
2. ejection of protein from solvent box

NOTE: add bond length/distance labels by pressing "2" on keyboard and left-clicking 2 atoms subsequently
![prod_md_frame49](B2/trial5_tleap_script_parm7_rst7/prod_md_frame49.png)\
NOTE: remove labels in `Graphics -> Labels -> Atoms/Bonds/...` (or right-clicking exactly on the left-clicked atoms) 
![prod_md_solvent_frame49](B2/trial5_tleap_script_parm7_rst7/prod_md_solvent_frame49.png)\
NOT a huge concern $\because$ based on my non-bonded cutoff value (9) (in angstroms),\
the closest residues (47.34) from proteins in adjacent periodic boxes (unit solvent cells) will not interact\
![prod_md_solvent_periodic_frame49](B2/trial5_tleap_script_parm7_rst7/prod_md_solvent_periodic_frame49.png)\
but NEXT time,
1. set non-bonded cutoff to 10, 
2. solvate the protein in a larger solvent box (>12),
3. preferably in cubic solvent box

MD trajectory movie `amber_o98_ion.parm7` + `o98_prod_md.nc`\
Find out how to generate the trajectory movie in MP4 or GIF!
```txt
<video src="B2/trial5_tleap_script_parm7_rst7/prod_md.mp4" controls>
Your browser does not support the video tag.
</video>
```

# USE OUTPUT TRAJECTORIES IN NEXT STEP (`05_.ipynb`)
Before 2024/06/18